In [1]:
import numpy as np
import pandas as pd
import math
import time
import random
import matplotlib.pyplot as plt

from collections import namedtuple,deque

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import BSS_ENV_shortduration as BE

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

KeyboardInterrupt: 

In [ ]:
#单一行为以及可拓展的策略
class options:
    def __init__(self,num):
        self.charger_num = num
        self.ev_num = num
        self.bssbatterynum = 2 * num
        self.maxpve = 0
        self.transit_step = 2
        self.ev_average_worksoc = 0.1

        self.inp_dim = 4
        self.memory_count = 1000
        self.GAMMA = 0.99
        self.beta = 0.2
        self.MEMORY_NUM = 2
        self.game_step = 400
        
        self.NUM_EPISODES = 2000

        self.update_gamecount = 4#每进行多少轮游戏更新网络
        
        self.learning_ratio = 0.0001
        self.if_clamp = True
        self.modelpath = r'D:\sjtu\EV_DRL\BSS_OPERATION_Emergence\data\savemodel\policymodel'
        self.dirpath=r'D:\sjtu\EV_DRL\BSS_OPERATION\data'
        self.gru_state = r'D:\sjtu\EV_DRL\BSS_OPERATION\data\savemodel\gru.pth'
        #self.price_data = r'D:\sjtu\xdd\paper\data\price\time_series_60min_singleindex.csv'
        self.price_datafile = r'D:\sjtu\EV_DRL\BSS_OPERATION_Emergence\data\pricedata\time_series_60min_singleindex.csv'
        self.price_data = self.pricedata()

        #模型的初始化格式
        self.pit = {'avg':0,'ban':1,'all':2}#等概率考虑pv，不使用pv，最大化使用pv

    def pricedata(self):
        price_data_path = self.price_datafile
        pd_frame = pd.read_csv(price_data_path)

        DK_1_price = pd_frame['DK_1_price_day_ahead']
        DK_1_price.fillna(0,inplace=True)

        DK_1_price_nparray =np.array([max(0,i) for i in DK_1_price.tolist()])
        return DK_1_price_nparray

def GetNewGame(opts:options):
    gmpricedata = opts.price_data
    gamemanager = BE.GameManager(opts,ps_eprice_ds=gmpricedata)
    #randomprice = [10 for _ in range(opts.MAX_TIME+1)] #[random.randint(1,20) for _ in range(opts.MAX_TIME+1)]
    return gamemanager       

OPTIONS = options(10)
GM = GetNewGame(OPTIONS)

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))
ax.plot(np.arange(1000), OPTIONS.price_data[0:1000], linewidth=1)
plt.show()

In [ ]:
def StateTensor(gm:BE.GameManager):
    bt1_num,bt2_num,b_num,energy_cost,carbon,bssevqueue_length,wasted_reserve,ctime,is_done = gm.EnvState()
    ctime = ctime/10 #0-96的数值太大，这里减少一下数值大小
    statetensor = torch.tensor([bt1_num,bt2_num,bssevqueue_length,ctime],dtype= torch.float32).to(device)
    costtensor = torch.tensor([energy_cost,bssevqueue_length,wasted_reserve],dtype= torch.float32).to(device)
    return statetensor.unsqueeze(0),costtensor.unsqueeze(0) #1*4 ,1*3

cfactor_t1 = 0.05
qfactor_t1 = 1
wfactor_t1 = 0.5
def rewardtype1(cost_T,opts:options):#正常运营状态下的奖励信息，车辆排队数量+超出的预约数+充电费用

    charging_cost = cfactor_t1* cost_T[:,0]/120#120为电池容量，这里电费最好按照SOC的比例计算，因此除掉了电池容量

    emergenceqfactor = 1 if opts.ev_num/cost_T[:,1]>1.5 else 3
    evqueuetimein_bss = qfactor_t1*(cost_T[:,1])*emergenceqfactor

    wast_request = wfactor_t1*cost_T[:,2]
       
    #print(evqueuetimein_bss, wast_request,charging_cost)
    bss_reward = -(evqueuetimein_bss+ wast_request+charging_cost)/opts.ev_num
    #print(torch.ones(1),bss_reward)
    bss_reward = bss_reward.unsqueeze(0) #奖励值输入形状为1*
    return bss_reward


def rewardtype1_detail(cost_T,opts:options):#正常运营状态下的奖励信息，车辆排队数量+超出的预约数+充电费用

    charging_cost = cfactor_t1* cost_T[:,0]/120#120为电池容量，这里电费最好按照SOC的比例计算，因此除掉了电池容量

    emergenceqfactor = 1 if opts.ev_num/cost_T[:,1]>1.5 else 3
    evqueuetimein_bss = qfactor_t1*(cost_T[:,1])*emergenceqfactor

    wast_request = wfactor_t1*cost_T[:,2]
       
    #print(evqueuetimein_bss, wast_request,charging_cost)
    bss_reward = -(evqueuetimein_bss+ wast_request+charging_cost)/opts.ev_num
    #print(torch.ones(1),bss_reward)
    bss_reward = bss_reward.unsqueeze(0) #奖励值输入形状为1*
    return bss_reward,-charging_cost,-evqueuetimein_bss,-wast_request

In [ ]:
workingtimes= range(400)
timereturn = 48
worktimefactor =   [1 * math.cos(i*math.pi/timereturn) + 1  + random.random()*0.1 for i in workingtimes] 
fig = plt.figure(figsize = (12,3))
plt.plot(workingtimes,worktimefactor)
plt.show()

In [ ]:
evqueuelengths = []
for i in range(200):#OPTIONS.game_step):
    reserve = int(OPTIONS.charger_num/2)
    chargeplan = 1
    GM.Update(reserve,chargeplan) 
    cstate,ccost = StateTensor(GM)
    evqueuelengths.append(ccost[:,1].cpu().item())
    #print(i,cstate,ccost)
    r = rewardtype1(ccost,OPTIONS)
    #print(r)
    #GM.Message()

fig = plt.figure(figsize = (12,3))
plt.plot(range(200),evqueuelengths)
plt.show()

In [ ]:
class PPONetwork(torch.nn.Module):
    def __init__(self,inpdim,reserve_dim=5,init_cpratio_id = 1):
        super(PPONetwork,self).__init__()
        self.inputlayer = torch.nn.Linear(inpdim,64).to(device)
        self.linear1 = torch.nn.Linear(64,128).to(device)

        #self.norm = torch.nn.BatchNorm1d(4).to(device)
        #self.relu = torch.nn.LeakyReLU().to(device)

        self.linear2 = torch.nn.Linear(128,64).to(device)

        self.out1 = torch.nn.Linear(64,reserve_dim).to(device)
        #self.out2 = torch.nn.Linear(64,chargeplan_dim).to(device)
        self.out3 = torch.nn.Linear(64,1).to(device)
        self.softmax = torch.nn.Softmax(dim=-1)
        
        
    def forward(self,inp):
        x = inp.to(device)
        inp1 = self.inputlayer(x)
        tanh1 = torch.nn.Tanh()(inp1)
        line1 = self.linear1(tanh1)
        line2 = self.linear2(line1)#(reshape_128)
        tanh2 = torch.nn.Tanh()(line2)
        #print(tanh2)
        out_1 = self.out1(tanh2)
        
        out_3 = self.out3(tanh2)
        res1 = self.softmax(out_1)

        #print(out_3)
        res3 = out_3
        return res1,res3 #bs*reservedim+1,bs*2,bs*1

In [ ]:
BSSTransition = namedtuple('BSSTransition',('state_pool','value_pool','act_pool','old_prob_pool','reward_pool','done_pool'))#收集连续动作
class BSSMemory(object):#策略梯度采样，每次保存一次策略的数据，采样一次策略的数据、
    def __init__(self,capacity):
        self.memory = deque([],maxlen = capacity)
    def push(self,*args):
        self.memory.append(BSSTransition(*args))
    def sample(self):
        batch = list(self.memory)
        return zip(*batch)
    def __len__(self):
        return len(self.memory)
    def clear(self):
        self.memory.clear()


class BSS_DRL(object):#er_dim=5*5+1默认单位电量20，5个100kwh充电桩对应的维度选择为25+1
    def __init__(self,opt):#ifstep 通过冻结网络的部分层操作实现分段训练):
        self.options =  opt
        self.lr = opt.learning_ratio
        
        self.bss_agent = PPONetwork(opt.inp_dim,reserve_dim=opt.charger_num +1,init_cpratio_id= 0).to(device)
        
        self.bss_agent_target = PPONetwork(opt.inp_dim,reserve_dim=opt.charger_num +1,init_cpratio_id=0).to(device)
        self.bss_agent_target.load_state_dict(self.bss_agent.state_dict())  
        self.bss_agent_target.eval()

        self.bss_agent_optimizer = torch.optim.Adam(self.bss_agent.parameters(),lr = self.lr)
        #self.bss_a2c_target_optimizer = torch.optim.Adam(self.evrp_critic.parameters(),lr = self.lr)
        self.if_clamp = opt.if_clamp
        self.bssmemory_num = opt.MEMORY_NUM
        self.bssmemorys = [] 
        for i in range(4):
            self.bssmemorys.append(BSSMemory(2500))
        
    #根据当前状态 选择车辆充电行为， 首先根据车辆策略模型给出期望充电行为， 再依据充电商策略模型给出确定动作   
    def select_action(self,bssstate,if_train=False):#state_s的格式为bs*ev_num*(state_n)
        prob1,state_value = self.bss_agent(bssstate)
        prob1_target,state_value_target = self.bss_agent_target(bssstate)
        
        reserve_dist = torch.distributions.categorical.Categorical(prob1)
   
        reserve_dist_target = torch.distributions.categorical.Categorical(prob1_target)
        
        
        reserve_act = reserve_dist.sample()

        reserve_act_target = reserve_dist_target.sample()
        
        
        if if_train:
            return reserve_act_target,reserve_dist_target,reserve_dist,state_value_target
                
        return reserve_act,prob1
    
    
    def cliprange(self,prob,prob_target,segma=0.2):
        policy_ratio = prob/prob_target
        max_clip = 1+segma
        min_clip = 1-segma
        clipres = torch.clamp(policy_ratio,min = min_clip,max = max_clip)
        return clipres
    
    def compute_returns(self,next_value, rewards, masks, gamma=0.99):
        value = next_value.detach()#return 不计入梯度计算，这里使用detach() 1*1
        rewards = (rewards - rewards.mean())/rewards.std()
       
        returns = torch.zeros_like(rewards)    #使用numpy操作可能更快？但是需要注意tensor和numpy数据的转换
        for act_i in reversed(range(len(rewards))):
            #print(rewards[act_i],gamma , value , masks[act_i])
            value = rewards[act_i] + gamma * value * (1-masks[act_i])
            returns[act_i] = value
        return returns #pool_size*1*1

    def update_bss(self,bssmemorys,nstate_s_list):
        #print(0.001*entropy)
        mmnum = len(bssmemorys)
        totallossvalue = 0
        inpoches = 6#每轮数据更新次数（正常情况下为每轮数据更新至梯度几乎消失 也就是过小后，不进行更新，或者过大的梯度跳过更新，随后才使用下一轮采集的数据）
        for _ in range(inpoches): #注意这里外循环为轮次，内循环为轨迹，这样可以每轮更新使用所有轨迹数据，避免单挑轨迹主导更新、
            losses = 0
            for i in range(self.bssmemory_num):
                'state_pool','value_pool','act_pool','old_prob_pool','reward_pool','done_pool'
                #tate_pool, value_pool, policy_ratio_pool, clipres_pool, reward_pool, done_pool = bssmemorys[i].sample()
                state_pool, value_pool, act_pool,old_prob_pool, reward_pool, done_pool = bssmemorys[i].sample()
                #在bs维度进行合并
                state_poolstack =torch.concat(state_pool,dim=0).to(device).detach()#pool_length*evnum*evdim #这里由于采样了一条轨迹的所有信息 所以一个轨迹的数据被合并到了bs维度
                old_value_poolstack = torch.concat(value_pool,dim=0).to(device).detach()#pool_length*1#记录的为整体的Q基于行为的平均状态价值
                act_poolstack = torch.concat(act_pool,dim=0).to(device).detach()#pool_length*evnum*1
                old_prob_poolstack = torch.concat(old_prob_pool,dim=0).to(device).detach()#pool_length*evnum*1
                #print(value_poolstack)
                reward_poolstack = torch.concat(reward_pool,dim=0).to(device)#pool_length*1 #整体的平均奖励
                #print(reward_poolstack)
                done_poolstack = torch.concat(done_pool,dim=0).to(device)#pool_length*1
                #print(done_poolstack)
                # print(state_poolstack.shape, old_value_poolstack.shape,\
                #         policy_ratio_poolstack.shape, clipres_poolstack.shape,\
                #             reward_poolstack.shape, done_poolstack.shape)
                

                current_prob_poolstack, current_bssqstatevalue  =self.bss_agent(state_poolstack)#bs*evnum*evactnum
                current_dist_poolstack = torch.distributions.categorical.Categorical(current_prob_poolstack)
                current_entropy = current_dist_poolstack.entropy().mean()
                              
                current_actprob_poolstack = torch.gather(current_prob_poolstack,dim=-1,index = act_poolstack)

                policy_ratio_poolstack = current_actprob_poolstack/old_prob_poolstack
                eps_clip = 0.2#clip范围值
                policy_clipres_poolstack = torch.clamp(policy_ratio_poolstack, 1-eps_clip, 1+eps_clip) #限制PPO 决策概率比值范围

                #1 1 1
                _,next_value = self.bss_agent_target(nstate_s_list[i])#bs*1
            
                #记录累加奖励（该采样与DQN不同，采样中每次仅记录一个策略动作序列）
                returns = self.compute_returns(next_value.squeeze(0),reward_poolstack,done_poolstack,gamma = self.options.GAMMA)
                advantages = returns - old_value_poolstack
                with torch.no_grad():#actor的梯度不需要考虑advantage内的梯度计算，因此这里归一化计算，且不计算梯度信息
                    adv_t = (advantages - advantages.mean()) / advantages.std().clamp_min(1e-6)

                _,current_value = self.bss_agent(state_poolstack)
                #print(current_value.shape,returns.shape)
                current_advantage = returns - current_value
  
                clip_advantages =  adv_t.detach() * policy_clipres_poolstack
                policy_ratio_advantages = advantages * policy_ratio_poolstack
                # print(clip_advantages)
                # print(policy_ratio_advantages)
                stack_advantages = torch.cat((clip_advantages,policy_ratio_advantages),dim=-1)
                ppo_advantages = stack_advantages.min(dim=-1)[0].unsqueeze(-1)
                
                #print( ppo_advantages)
                #print(advantages)
                #print(log_prob_poolstack)
                #print(advantages)
                
                actor_loss = -ppo_advantages.mean() # (-log_prob_poolstack * advantages).mean()
                critic_loss = 0.5 * current_advantage.pow(2).mean()
                # print('bss动作损失：{}'.format(actor_loss))
                # print('bss策略损失：{}'.format(critic_loss))
                total_loss = 10* actor_loss + critic_loss - 0.0001* current_entropy
                losses+= total_loss
            
            mmloss = losses/mmnum
            lossvalue = mmloss.cpu().detach().item()
            totallossvalue += lossvalue
           
            #print('动作熵值：{}'.format(log_prob_poolstack))            
            self.bss_agent_optimizer.zero_grad()      
            mmloss.backward()
            for name, param in self.bss_agent.named_parameters():
                #print(name,param.grad)
                if param.grad is not None:
                    param.grad.clamp_(-1,1)
            #for param in self.evrp_critic.parameters():           
            # print(param.grad)
            self.bss_agent_optimizer.step()    
        return totallossvalue/inpoches

    def freshtargetmodel(self):
        self.bss_agent_target.load_state_dict(self.bss_agent.state_dict())   

    def savemodel(self,dicpath,addname):
        torch.save(self.bss_agent.state_dict(),os.path.join(dicpath,addname+'.pth'))


In [ ]:
bss_drl = BSS_DRL(OPTIONS)

reward = rewardtype1


emergencefactor = 2.
bss_mmnum = 4 #PPO每次更新的轨迹采样次数


bss_memorys = [] 
for i in range(bss_mmnum):
    bss_memorys.append(BSSMemory(2500))
'current_state','action','reward','next_state','is_done'


In [ ]:
#bss_drl.bss_agent.load_state_dict(torch.load(os.path.join(OPTIONS.modelpath,'PPO_SCALE10_singleact_metamodelV1type4.pth')))
METAPATH = os.path.join(OPTIONS.modelpath,'PPO_SCALE10_singleact_metamodelREPTILE20250414.pth')
#METAPATH = os.path.join(OPTIONS.modelpath,'PPO_SCALE10_singleact_metamodelV1type4.pth')
METASTAT = torch.load(METAPATH ,weights_only=True)
bss_drl.bss_agent.load_state_dict(METASTAT)

In [ ]:
random_step = 200
bss_ep_rewards = []
for i in range(2000):# (OPTIONS.NUM_EPISODES+1000): # 
    start = time.time()#记录每个episode训练时间         
    bss_ep_entropys = [] 
    bss_last_state_s=[]
    bss_ep_reward = 0 #记录每个episode的平均奖励           
    
    for bss_mmnum_k1 in range(bss_mmnum): #submodel 先更新一次
        GM = GetNewGame(OPTIONS) 
                
        for reqstage_k in range(OPTIONS.game_step):

            cstate,ccoststate = StateTensor(GM)
            
            #BSSs决策过程
            reserve_act_target,reserve_dist_target,reserve_dist,state_value_target = bss_drl.select_action(cstate,True)

            reserve_act_target = reserve_act_target.unsqueeze(-1)
            bss_probs_target = reserve_dist_target.probs#1*evnum*evactnum
            bss_actprob_target =torch.gather(bss_probs_target,dim=-1,index=reserve_act_target)#1*evnum*1

                    #根据动作更新环境
            charge_request = reserve_act_target.cpu().item()

            GM.Update(charge_request,0,emergencefactor)

            nstate,ncoststate = StateTensor(GM)

            bss_reward = reward(ncoststate,OPTIONS) #1,1
            bss_reward = bss_reward.detach()
            bss_ep_reward += float(bss_reward)

            bss_nstate = nstate.detach()#同时需要添加到replaybuffer里面           
            
            'state_pool','value_pool','act_pool','old_prob_pool','reward_pool','done_pool'
            #1*bssnum*bssdim
            bss_cstate = cstate.detach()
            bss_cstatevalue = state_value_target.detach()
            bss_act_target =  reserve_act_target.detach()
            bss_act_prov = bss_actprob_target.detach()
            bss_rew = bss_reward

            isdone = torch.zeros(1,1).detach()#不考虑结束状态
            bss_memorys[bss_mmnum_k1].push(\
                bss_cstate,bss_cstatevalue,bss_act_target,\
                    bss_act_prov,bss_rew,isdone)
            
          
        nev_P = bss_nstate #只记录最后时刻
        bss_last_state_s.append(nev_P)

    #更新子任务模型 并重置参数信息
    bsslossvalue = bss_drl.update_bss(bss_memorys,bss_last_state_s)
    for bssmm in bss_memorys:
        bssmm.clear() 
    
    bss_ep_reward  = bss_ep_reward /bss_mmnum  #每局平均奖励
    
    bss_ep_rewards.append(bss_ep_reward)    
    
        
    if i%3==0:
        print('更新old网络')
        bss_drl.bss_agent_target.load_state_dict(bss_drl.bss_agent.state_dict())
        
        
    print ('第{}轮训练用时{:.3f}秒，平均累积奖励为{:.3f}'.format(
        i + 1, time.time()-start,bss_ep_reward))



    #valicationmessage(24)
#('current_state','goal_action','ev_prob','reward','next_state'))  

In [ ]:
fig = plt.figure(figsize = (12,6))
x = range(len(bss_ep_rewards))
ep_rewards_arr = bss_ep_rewards
plt.plot(x,ep_rewards_arr)
plt.show()

In [ ]:
ep_rewards_pd = pd.DataFrame(ep_rewards_arr)
pdpath = r'D:\sjtu\EV_DRL\BSS_OPERATION_Emergence\data\实验数据\训练数据\PPO_emgfactor200v1_SCALE10_singleact_trainreturn_metafinetuningv1.csv'
ep_rewards_pd.to_csv(pdpath,header=['return'],index=None)

In [ ]:
def test(step,gamemanager:BE.GameManager,rewardF,opt, model=None,showmessage = True,savepic = False):
    gamemanager.Initial( reinitial= True)
    print('仿真开始')
    #gamemanager.Message()
    request_reserve_list = []
    request_engs= []
    request_Gengs = []
    request_PVengs= []
    cplan_list = []
    
    wait_nums = []
    ava_bs = []
    gprices = []
    
    rew_list = []
    rcosts = []
    rqueues = []
    rwasts = []



    cplan_probs = []



    readybT1s =[]
    readybT2s = []
    cplans = []

    rewards = 0
    gamestep = 0
    for i in range(step):
        gamestep += 1
        csta,ccost = StateTensor(gamemanager)

        #BSS决策过程
        reserve_act,prob1 = opt.charger_num / 2,  None
        if  model != None:
            reserve_act,prob1 =  model.select_action(csta)
            #energy_req = energy_act.cpu().item()*er_factor if type_e else int(energy_act.cpu().item())
            reserve_act = prob1.argmax().cpu().item()
            #pv_rat = pv_act.cpu().item()
            
        # print(csta)
        # print(prob1)
        # print(prob2)
        # print(reserve_act,cplan_act)


        
        gamemanager.Update(reserve_act,0,emergencefactor,showmessage)
        request_engs.append(gamemanager.stage_used_energy)
        
        nsta,ncost = StateTensor(gamemanager)

        print(ncost)

        rew,rcost,rqueue,rwast =rewardF(ncost,opt)

        wait_num = gamemanager.bss_queuelength
        wait_nums.append(wait_num)

        gprices.append(gamemanager.ps.price)
        

        avab = gamemanager.ava_bnum#可获得的电池数
        ava_bs.append(avab)

        request_reserve_list.append(reserve_act)

        
        
       
       
        
        rew_list.append(rew.cpu().item())
        rcosts.append(rcost.cpu().item())
        rqueues.append(rqueue.cpu().item())
        rwasts.append(rwast.cpu().item())



        rewards += rew

        readybT1s.append(gamemanager.bss.BT1_readdy_num)
        readybT2s.append(gamemanager.bss.BT2_readdy_num)


        if showmessage:
            gamemanager.Message()

    #print(request_energy_list)

    def plotax(axsi,x,y,names):
        lines = []
        for i in y:
            li, = axsi.plot(x, i)
            lines.append(li)
        axsi.legend(lines,names)

    fig,axs = plt.subplots(6,1,figsize=(12,3*6))

    x = range(gamestep)
    plotax(axs[0],x,[request_reserve_list],['request_reserve'])
    # axs[0].plot(x,request_energy_list)
    # axs[0].plot(x,pvenergies)
    plotax(axs[1],x,[wait_nums,ava_bs],['wait_nums','ava_bs'])
    # axs[1].plot(x,wait_nums)
    # axs[1].plot(x,ava_bs)

    plotax(axs[2],x,[gprices],['gprices'])
    # axs[2].plot(x,gprices)
    # axs[2].plot(x,pvprices)

    plotax(axs[3],x,[request_engs],['eng','Geng','pveng'])
    #axs[3].plot(x, [min(i,200) for i in pveng])
    

    #axs[4].plot(x, maxpvs)#pv_probs)#pv_list)

    plotax(axs[4],x,[readybT1s,readybT2s],['readybT1s','readybT2s'])
    # axs[5].plot(x, readybT1s)#pv_probs)#pv_list)
    # axs[5].plot(x, readybT2s)#pv_probs)#pv_list)
    # axs[5].plot(x, cplans)
    
    #plotax(axs[6],x,[rew_list],['rew_list'])
    plotax(axs[5],x,[rcosts,rqueues,rwasts],['rcost','rqueue','rwast'])
    axs[5].plot(x,rew_list)
        
    if savepic:
        path = os.path.join(opt.dirpath,'a2csimulation'+opt.t1+'.png')
        plt.savefig(path)

    plt.show()
    return rew_list

In [ ]:
emergencefactor

In [ ]:
rews =test(400,gamemanager=GM,rewardF=rewardtype1_detail,opt= OPTIONS,
                     model=bss_drl,showmessage=True,savepic=False)
sum(rews)

In [ ]:
torch.save(bss_drl.bss_agent.state_dict(),f=r'D:\sjtu\EV_DRL\BSS_OPERATION_Emergence\data\savemodel\policymodel\PPO_emgfactor050v1_SCALE10_singleact.pth')